# Forecasting Model for Restaurant Operations

This notebook builds a production-style forecasting system for branch-level revenue and profit trends, providing actionable business insights for operational planning, revenue forecasting, and risk anticipation.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

# For MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load the data
df = pd.read_csv('data/daily_operations.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['restaurant_id', 'date']).reset_index(drop=True)
print(f"Data loaded: {df.shape[0]} rows")
print(df.head())

In [ ]:
# Check for missing dates
all_dates = pd.date_range(start=df['date'].min(), end=df['date'].max())
missing_dates = all_dates.difference(df['date'].unique())
if len(missing_dates) > 0:
    print(f"Missing dates: {len(missing_dates)}")
    # Forward fill or interpolate, but for simplicity, assume no missing
else:
    print("No missing dates")

# Group by branch
branches = df['restaurant_id'].unique()
print(f"Number of branches: {len(branches)}")

In [ ]:
# Prepare monthly profit data
df['month'] = df['date'].dt.to_period('M')
monthly_profit = df.groupby(['restaurant_id', 'month'])['net_profit'].sum().reset_index()
monthly_profit['month'] = monthly_profit['month'].dt.to_timestamp()
print("Monthly profit aggregated")
print(monthly_profit.head())

In [ ]:
# Train-test split: last 3 months for test
test_months = 3
max_date = df['date'].max()
test_start = max_date - pd.DateOffset(months=test_months)
train_df = df[df['date'] < test_start]
test_df = df[df['date'] >= test_start]
print(f"Train period: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"Test period: {test_df['date'].min()} to {test_df['date'].max()}")

In [ ]:
# Baseline 1: Moving Average (7-day)
def moving_average_forecast(series, window=7):
    return series.rolling(window=window).mean()

# Baseline 2: Linear Trend
def linear_trend_forecast(series):
    x = np.arange(len(series)).reshape(-1, 1)
    y = series.values
    model = LinearRegression()
    model.fit(x, y)
    trend = model.predict(x)
    # For forecast, extend
    future_x = np.arange(len(series), len(series) + len(series)).reshape(-1, 1)
    forecast = model.predict(future_x)
    return pd.Series(trend, index=series.index), pd.Series(forecast, index=series.index + pd.Timedelta(days=len(series)))

# Advanced: ARIMA
def arima_forecast(series, order=(1,1,1)):
    model = ARIMA(series, order=order)
    fitted = model.fit()
    forecast = fitted.forecast(steps=len(series))
    return fitted.fittedvalues, forecast

print("Forecasting functions defined")

In [ ]:
# Evaluate for all branches
evaluation_results = []
for branch_id in branches:
    branch_data = train_df[train_df['restaurant_id'] == branch_id].set_index('date')['revenue']
    test_branch = test_df[test_df['restaurant_id'] == branch_id].set_index('date')['revenue']
    if len(branch_data) < 14 or len(test_branch) == 0:
        continue  # Skip if not enough data

    # Moving Average
    ma_pred = moving_average_forecast(branch_data, window=7)
    ma_test_pred = ma_pred.reindex(test_branch.index).fillna(method='ffill')

    # Linear Trend
    trend_pred, trend_forecast = linear_trend_forecast(branch_data)
    trend_test_pred = trend_forecast.head(len(test_branch)).values

    # ARIMA
    try:
        arima_pred, arima_forecast_vals = arima_forecast(branch_data)
        arima_test_pred = arima_forecast_vals[:len(test_branch)]
    except:
        arima_test_pred = np.full(len(test_branch), np.nan)

    # Calculate metrics
    ma_metrics = calc_metrics(test_branch, ma_test_pred)
    trend_metrics = calc_metrics(test_branch, trend_test_pred)
    arima_metrics = calc_metrics(test_branch, arima_test_pred)

    evaluation_results.append({
        'branch_id': branch_id,
        'MA_MAE': ma_metrics['MAE'],
        'Trend_MAE': trend_metrics['MAE'],
        'ARIMA_MAE': arima_metrics['MAE'],
        'best_model': min([('MA', ma_metrics['MAE']), ('Trend', trend_metrics['MAE']), ('ARIMA', arima_metrics['MAE'])], key=lambda x: x[1] if not np.isnan(x[1]) else float('inf'))[0]
    })

eval_df = pd.DataFrame(evaluation_results)
print(eval_df.head())

In [ ]:
# Visualization for one branch
plt.figure(figsize=(12, 6))
plt.plot(branch_data.index, branch_data.values, label='Actual Train')
plt.plot(test_branch.index, test_branch.values, label='Actual Test')
plt.plot(test_branch.index, trend_test_pred, label='Trend Forecast')
plt.plot(test_branch.index, arima_test_pred, label='ARIMA Forecast')
plt.title(f'Branch {branch_id} Revenue Forecast')
plt.legend()
plt.show()

In [ ]:
# Monthly Profit Forecasting
# Prepare monthly data
train_monthly = monthly_profit[monthly_profit['month'] < test_start.to_period('M').to_timestamp()]
test_monthly = monthly_profit[monthly_profit['month'] >= test_start.to_period('M').to_timestamp()]

# For each branch, forecast monthly profit
monthly_forecasts = []
for branch in branches:
    branch_monthly = train_monthly[train_monthly['restaurant_id'] == branch].set_index('month')['net_profit']
    if len(branch_monthly) < 6:  # Need at least 6 months
        continue
    # Simple trend for monthly
    x = np.arange(len(branch_monthly)).reshape(-1, 1)
    y = branch_monthly.values
    model = LinearRegression().fit(x, y)
    future_x = np.arange(len(branch_monthly), len(branch_monthly) + 3).reshape(-1, 1)  # Forecast 3 months
    forecast = model.predict(future_x)
    monthly_forecasts.append({'branch_id': branch, 'forecast_profit': forecast.mean()})

monthly_df = pd.DataFrame(monthly_forecasts)
print("Monthly profit forecasts:")
print(monthly_df.head())

In [ ]:
# Business Interpretation
# For each branch, determine trend and forecasts
results = []
for branch in branches:
    branch_series = df[df['restaurant_id'] == branch].set_index('date')['revenue']
    branch_monthly = monthly_profit[monthly_profit['restaurant_id'] == branch]
    # Revenue trend
    x = np.arange(len(branch_series)).reshape(-1, 1)
    y = branch_series.values
    model = LinearRegression().fit(x, y)
    slope = model.coef_[0]
    direction = 'up' if slope > 100 else 'down' if slope < -100 else 'stable'  # Threshold for significance
    next_revenue = model.predict([[len(branch_series)]])[0]
    current_revenue = branch_series.mean()
    revenue_pct_change = ((next_revenue - current_revenue) / current_revenue) * 100
    # Profit: from monthly
    if len(branch_monthly) > 0:
        current_profit = branch_monthly['net_profit'].mean()
        # Forecast profit
        profit_forecast = monthly_df[monthly_df['branch_id'] == branch]['forecast_profit'].values
        if len(profit_forecast) > 0:
            profit_pct_change = ((profit_forecast[0] - current_profit) / current_profit) * 100
        else:
            profit_pct_change = np.nan
    else:
        profit_pct_change = np.nan
    # Risk: volatility
    volatility = branch_series.std()
    risk = 'high' if volatility > current_revenue * 0.5 else 'medium' if volatility > current_revenue * 0.3 else 'low'
    results.append({
        'branch_id': branch, 
        'trend_direction': direction, 
        'forecasted_revenue_change_%': revenue_pct_change,
        'forecasted_profit_change_%': profit_pct_change,
        'risk_flag': risk
    })

summary_df = pd.DataFrame(results)
summary_df = summary_df.sort_values(by='forecasted_revenue_change_%', ascending=True)  # Highest risk (declining)
print(summary_df.head(10))

## Operational Forecast Summary

The table above provides a structured summary of forecasted trends, revenue and profit changes, and risk flags per branch. Branches showing declining trends or high risk require immediate attention for operational planning:

- **Upward Trends**: Capitalize on growth by increasing staffing and supplies.
- **Declining Trends**: Investigate causes (e.g., competition, economic factors) and implement cost controls or marketing campaigns.
- **Stable but Low-Performing**: Optimize efficiency to improve margins.
- **High Risk**: Monitor closely for volatility; prepare contingency plans.

This decision support tool answers: "What will happen next, where, and what should operations do about it?"